# Search source by meaning

Tutorial 60 compared a question vector with chunk vectors directly. This
tutorial turns that idea into reusable **semantic indexing, retrieval, and
ranking**.

- **Indexing:** make each repository chunk searchable by storing its embedding.
- **Retrieval:** compare a new question with those stored chunk embeddings.
- **Ranking:** order the chunks from highest to lowest cosine similarity.

No chat model writes an answer yet. Tutorial 80 adds lexical retrieval for
exact words and identifiers.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load the embedding model

Semantic retrieval uses the same embedding model family as tutorial 60.
`auto` selects MLX on Apple Silicon and PyTorch elsewhere.


In [ ]:
from qachat.runtime import choose_backend

BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(selected_backend, MODEL_ID)


In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare repository chunks

These are the same lower layers used in tutorials 40–60: select files, read
them, extract natural source units, and split only units that are too large.


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py", ".md"],
    include_paths=["qachat", "README.md"],
)

print(f"Selected files: {len(report['selected_files'])}")


In [ ]:
from qachat.sources.sources import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Read files: {len(source_files)}")
print(f"Natural units: {len(units)}")


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Build the semantic index

The **index** is prepared before a user searches. Each chunk is embedded once,
and the vector is stored beside that chunk. Later questions reuse these stored
vectors instead of embedding all repository chunks again.

This tutorial uses a simple in-memory index so the mechanism stays visible.


In [ ]:
from qachat.retrieval import build_semantic_index

semantic_index = build_semantic_index(
    chunks,
    embedding_session,
)

print(f"Indexed chunks: {len(semantic_index.chunks)}")
print(f"Vector size: {len(semantic_index.embeddings[0])}")


In [ ]:
first_chunk = semantic_index.chunks[0]
first_vector = semantic_index.embeddings[0]

print(f"Chunk ID: {first_chunk.identifier}")
print(f"Source: {first_chunk.path}")
print(f"Stored vector starts: {first_vector[:5]}")


## 4. Search the index

A new question is embedded once. `semantic_search` compares that query vector
with every stored chunk vector using cosine similarity, then sorts the scores.

The question below deliberately does not copy a function name. Semantic
retrieval should still find code about choosing the computation backend.


In [ ]:
from qachat.query_formats import (
    SOFTWARE_REPOSITORY_INSTRUCTION,
    format_qwen3_query,
)

QUESTION = "How does QAChat decide which computation backend to run on?"

query_text = format_qwen3_query(
    question=QUESTION,
    instruction=SOFTWARE_REPOSITORY_INSTRUCTION,
)

print(query_text)


In [ ]:
from qachat.retrieval import semantic_search

results = semantic_search(
    semantic_index,
    query_text,
    embedding_session,
    count=5,
)


## 5. Inspect the ranking

Higher cosine similarity means the chunk is semantically closer to the
question. The score is useful for **ordering** candidates; it is not a
probability that a chunk is correct.


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"{rank}. {result.score:.3f} | {location} | {name}"
    )


In [ ]:
best = results[0]

print("Most relevant chunk:")
print()
print(best.chunk.text)


## 6. What happened?

For repository chunks, indexing happened **once**:

`chunk -> embedding -> semantic index`

For this question, retrieval happened **at search time**:

`question -> embedding -> cosine similarity -> ranking -> top chunks`

Tutorial 80 uses a different index for **lexical retrieval**, where exact
words and software identifiers matter more than semantic meaning.


## 7. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Result

QAChat now has a reusable semantic index: repository chunks are embedded once,
a question is compared with those stored vectors, and the closest meanings are
ranked and returned. No LLM answer generation is involved yet.
